# Silver Layer: Wind Turbine Data Cleaning

Reads from `bronze.wind_turbine_raw`, applies null handling and outlier removal, and writes cleaned rows to `silver.wind_turbine_clean`.

## Cleaning steps

### 1. Null handling
| Column | Strategy | Rationale |
|---|---|---|
| `timestamp` | **Drop row** | A reading with no time context is unplaceable in any time-series analysis |
| `turbine_id` | **Drop row** | Cannot attribute a reading to any turbine |
| `power_output_mw` | **Drop row** | Core metric — imputing power from wind speed would introduce modelling assumptions outside the pipeline's scope |
| `wind_speed_ms` | **Impute with per-turbine daily median** | Commonly available from nearby sensors; median is robust to remaining outliers |
| `wind_direction_deg` | **Impute with per-turbine daily median** | Same rationale as wind speed |

### 2. Outlier detection and removal
Outliers are defined per-turbine using a rolling **±2 standard deviation** window over each numeric column.  
Rows where any numeric column falls outside `[mean − 2σ, mean + 2σ]` are flagged with `_is_outlier = true` and excluded from the clean output.

Stats are computed **per turbine per day** to account for naturally different operating baselines across turbines and diurnal wind patterns.

### 3. Physical range guards
Hard physical limits are applied as a first pass before statistical outlier detection:
- `wind_speed_ms` ∈ [0, 50] — typical cut-out speed is ~25 m/s; 50 is a generous upper bound
- `wind_direction_deg` ∈ [0, 360]
- `power_output_mw` ≥ 0 — negative power is physically impossible

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "wind_farm_dev")
dbutils.widgets.text("schema",  "wind_farm")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

bronze_table = f"`{catalog}`.`{schema}`.wind_turbine_raw"
silver_table = f"`{catalog}`.`{schema}`.wind_turbine_clean"

print(f"Source : {bronze_table}")
print(f"Target : {silver_table}")

In [ ]:
from pyspark.sql import functions as F, Window

bronze_df = spark.table(bronze_table)
print(f"Bronze row count: {bronze_df.count():,}")

## Step 1 — Drop rows with null anchor columns

In [ ]:
from wind_farm.utils.turbine import TurbineCleaner as tc

after_anchor_drop_df = bronze_df.dropna(subset=tc.ANCHOR_COLS)

dropped_anchor = bronze_df.count() - after_anchor_drop_df.count()
print(f"Rows dropped (null anchor column): {dropped_anchor:,}")

## Step 2 — Physical range filter

In [ ]:

dropped_range_df = tc.range_filter(after_anchor_drop_df)

dropped_range = after_anchor_drop_df.count() - dropped_range_df.count()
print(f"Rows dropped (outside physical bounds): {dropped_range:,}")

## Step 3 — Impute remaining nulls with per-turbine daily median

In [ ]:
imputed_df = tc.impute_nulls(dropped_range_df)

imputed = dropped_range_df.count() - imputed_df.count()
print(f"Rows dropped (un-imputable nulls after median fill): {imputed:,}")

## Step 4 — Statistical outlier detection (±2σ per turbine per day)

In [ ]:
filtered_df = tc.filter_outliers(imputed_df)

outlier_count = filtered_df.filter(F.col("_is_outlier")).count()
print(f"Outlier rows flagged: {outlier_count:,}")

## Step 5 — Build silver output

Drop all intermediate stat/flag columns and exclude outlier rows from the clean table.

In [ ]:
# Final clean dataframe, with outliers removed and renamed columns for clarity
silver_df = (
    filtered_df
        .filter(~F.col("_is_outlier"))
        .drop("_is_outlier")
        .withColumn("_cleaned_at", F.current_timestamp())
        .withColumnRenamed("wind_speed", "wind_speed_ms")
        .withColumnRenamed("wind_direction", "wind_direction_deg")
        .withColumnRenamed("power_output", "power_output_mw")
)

print(f"Silver row count: {silver_df.count():,}")

## Step 6 — Write silver Delta table

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

display(silver_df)

# (
#     silver_df.write
#         .format("delta")
#         .mode("overwrite")              # Full reprocess on each run
#         .option("overwriteSchema", "true")  
#         .saveAsTable(silver_table)
# )

# print(f"Silver write complete → {silver_table}")

## Step 7 — Data quality summary

In [ ]:
bronze_count = bronze_df.count()
silver_count = spark.table(silver_table).count()
retention_pct = 100 * silver_count / bronze_count if bronze_count else 0

print("=" * 50)
print(f"Bronze rows              : {bronze_count:>10,}")
print(f"  Dropped — null anchor  : {dropped_anchor:>10,}")
print(f"  Dropped — range bounds : {dropped_range:>10,}")
print(f"  Dropped — un-imputable : {dropped_impute:>10,}")
print(f"  Dropped — outliers     : {outlier_count:>10,}")
print(f"Silver rows              : {silver_count:>10,}  ({retention_pct:.1f}% retained)")
print("=" * 50)

print("\nNull counts remaining in silver:")
spark.table(silver_table).select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in NUMERIC_COLS + ["timestamp", "turbine_id"]]
).show()